In [2]:
import dotenv
%load_ext dotenv
%dotenv

import os
import platform
import json
import random
import time

import jax
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from absl import app, flags
from ml_collections import config_flags

from agents import agents
from envs.env_utils import make_env_and_datasets
from utils.datasets import Dataset, ReplayBuffer
from utils.evaluation import evaluate, flatten, supply_rng
from utils.flax_utils import restore_agent, save_agent

import jax
import numpy as np
import matplotlib.pyplot as plt

def supply_rng(f, rng=jax.random.PRNGKey(0)):
    """Helper function to split the random number generator key before each call to the function."""

    def wrapped(*args, **kwargs):
        nonlocal rng
        rng, key = jax.random.split(rng)
        return f(*args, seed=key, **kwargs)

    return wrapped

# Configuration
env_name = "antmaze-large-navigate-singletask-task1-v0"
seed = 0
buffer_size = 2000000
eval_episodes = 50

# Agent paths and epochs
fbrac_task1_path="/mnt/nas/jaehyeok/fql/exp/fql/Debug/fbrac_antmaze-large-navigate-singletask-task1-v0_sd000_20250907_031541_utd-ratio1"

# agent_path=a5_jacb_reg_e5_agent_path
agent_path=fbrac_task1_path
restore_epoch=1000000

# double check the agent seed
def extract_seed(agent_path):
    # extract sd{:3}_ from agent_path
    return int(agent_path.split("_sd")[1].split("_")[0])
agent_seed = extract_seed(agent_path)

if agent_seed != seed:
    print(f"Agent seed {agent_seed} does not match the seed {seed}")
else:
    # Make environment and datasets
    env, eval_env, train_dataset, val_dataset = make_env_and_datasets(env_name, frame_stack=None)
    train_dataset = Dataset.create(**train_dataset)
    train_dataset = ReplayBuffer.create_from_initial_dataset(
        dict(train_dataset), size=max(buffer_size, train_dataset.size + 1)
    )

    # Initialize random seeds
    random.seed(seed)
    np.random.seed(seed)
    example_batch = train_dataset.sample(1)

    from agents.fbrac import get_config as get_fbrac_config
    from agents.fql import get_config as get_fql_config
    fbrac_config = get_fbrac_config()
    fql_config = get_fql_config()

    fbrac_agent_class = agents[fbrac_config['agent_name']]
    fbrac_agent = fbrac_agent_class.create(
        seed,
        example_batch['observations'],
        example_batch['actions'],
        fbrac_config,
    )

    # Restore trained models
    if agent_path and restore_epoch:
        fbrac_agent = restore_agent(fbrac_agent, agent_path, restore_epoch)
        print(f"FBRAC agent restored from {agent_path} at epoch {restore_epoch}")

    print("Models loaded successfully!")


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
Restored from /mnt/nas/jaehyeok/fql/exp/fql/Debug/fbrac_antmaze-large-navigate-singletask-task1-v0_sd000_20250907_031541_utd-ratio1/params_1000000.pkl
FBRAC agent restored from /mnt/nas/jaehyeok/fql/exp/fql/Debug/fbrac_antmaze-large-navigate-singletask-task1-v0_sd000_20250907_031541_utd-ratio1 at epoch 1000000
Models loaded successfully!


In [3]:
# Generate trajectories for both agents
print("Generating trajectories for FBRAC agent...")
eval_info, trajs, renders = evaluate(
    agent=fbrac_agent,
    env=eval_env,
    config=fbrac_config,
    num_eval_episodes=eval_episodes,
    num_video_episodes=0,  # Skip video rendering for now
)

eval_info

Generating trajectories for FBRAC agent...


100%|██████████| 50/50 [00:34<00:00,  1.44it/s]


defaultdict(list,
            {'xy': 29.482882318469052,
             'prev_qpos': 4.021247866249551,
             'prev_qvel': 0.11826780494619314,
             'qpos': 4.02879435132694,
             'qvel': 0.02854931410791663,
             'success': 0.92,
             'total.timesteps': 15335.32,
             'episode.final_reward': -0.08,
             'episode.return': -604.0,
             'episode.length': 604.92,
             'episode.duration': 0.6933101558685303})

In [25]:
import jax
import jax.numpy as jnp

def analyze_flow_actor_sensitivity(agent, batch, actor_name='actor_bc_flow', sensitivity_type='integrated'):
    """
    Analyze actor sensitivity for flow matching agents.
    
    Args:
        agent: Flow matching agent (FQL, IFQL, etc.)
        batch: Batch containing observations
        sensitivity_type: 'velocity' or 'integrated'
    """
    assert sensitivity_type in ['velocity', 'integrated']

    observations = batch['observations']
    
    if sensitivity_type == 'velocity':
        # Sample random actions and times for velocity field analysis
        batch_size, action_dim = observations.shape[0], agent.config['action_dim']
        actions = jax.random.normal(jax.random.PRNGKey(0), (batch_size, action_dim))
        times = jax.random.uniform(jax.random.PRNGKey(1), (batch_size, 1))
        
        def velocity_fn(state):
            return agent.network.select(actor_name)(state[None, :], actions[None, :], times[None, :])[0]
        
        jacobians = jax.vmap(jax.jacobian(velocity_fn))(observations)
        
    elif sensitivity_type == 'integrated':
        # Analyze sensitivity of final integrated actions
        def integrated_fn(state):
            return agent.sample_actions(state[None, :], seed=jax.random.PRNGKey(0))[0]
        
        jacobians = jax.vmap(jax.jacobian(integrated_fn))(observations)
    
    # Compute sensitivity metrics
    sensitivity_metrics = {
        'jacobian_norm': jnp.linalg.norm(jacobians, axis=(1, 2)),  # Per-sample sensitivity
        'state_sensitivity': jnp.linalg.norm(jacobians, axis=1),   # Per-state-dimension sensitivity
        'action_sensitivity': jnp.linalg.norm(jacobians, axis=2),  # Per-action-dimension sensitivity
        'max_sensitivity': jnp.max(jnp.abs(jacobians), axis=(1, 2))  # Maximum sensitivity
    }
    
    return jacobians, sensitivity_metrics

In [26]:
batch_size = 256
batch = train_dataset.sample(batch_size)

analyze_flow_actor_sensitivity(
    fbrac_agent, 
    batch,
    actor_name='actor_bc_flow',
    sensitivity_type='velocity')

TypeError: Cannot concatenate arrays with different numbers of dimensions: got (256, 1, 29), (256, 1, 256, 8), (256, 1, 256, 1).